In [1]:
building = 0
steps = 1

In [2]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math

import Project.src.data.dataprep as prep

from Project.src.models.lstm import LSTM
from Project.src.models.lstmopt import LSTMOPT

import Project.src.tensors.tensorisation as tensor
import Project.src.optimization.pv_battery as pvb

def torch_py(torch_tensor):
    return torch_tensor.cpu().detach().numpy().flatten()

def rescale(values, scaler):
    rescaled_values = values * (scaler[1] - scaler[0]) + scaler[0]   
    return rescaled_values

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
# Base data
nl_data = prep.dutch_data('../data/Dutchdata_clean/building_' + str(building) + '.parquet', 'h')
nl_data['net_load'] = nl_data['load'] - nl_data['solar_energy']
nl_price_data = pd.read_csv('../data/NL_DA_Prices.csv', index_col='Date',parse_dates=True, dayfirst=True)
nl_data = nl_data.merge(nl_price_data[['offtake', 'injection']], left_index=True, right_index=True)
nl_data['cost'] = nl_data.apply(lambda row: row['net_load'] * row['injection'] if row['net_load'] < 0 else row['net_load'] * row['offtake'], axis=1)

In [5]:
# Base parameters
battery_capacity = round(nl_data['solar_energy'].max() * 1.24,2)

max_charge = battery_capacity/2.7
max_discharge = max_charge

layers = 3
neurons = 200

In [6]:
pvb_system = pvb.PV_battery(nl_data, battery_capacity, max_charge, max_discharge)

In [7]:
initial_battery_lstm = []
imported_energy_lstm = []
exported_energy_lstm = []
imported_energy_lstm_real = []
exported_energy_lstm_real = []

initial_battery_cvx = []
imported_energy_cvx = []
exported_energy_cvx = []
imported_energy_cvx_real = []
exported_energy_cvx_real = []

initial_battery_naive = []
imported_energy_naive = []
exported_energy_naive = []
imported_energy_naive_real = []
exported_energy_naive_real = []

initial_battery_perfect = []
imported_energy_perfect = []
exported_energy_perfect = []

lstm_pv = []
cvx_pv = []

T = 24
lags = 24
forecast_gap = 0
last_opt = T - steps

for t in range(T,last_opt,-1):
       
    print('Setting up optimization for hour ' + str(t))
        
    problem, variables, parameters = pvb_system.create_optimization_problem(t)
    problem_post, variables_post, parameters_post = pvb_system.create_post_forecast_optimization_problem(t)
    
    # Tensors for training a base forecaster

    tensors = tensor.Tensors(pvb_system.house,'solar_energy',['solar_energy'],[],lags,t, forecast_gap=forecast_gap)
    _, X_test, _, y_test, scalers = tensors.create_tensor()
    
    # Tensors for training an E2E network
    tensors_opt = tensor.Tensors(pvb_system.house,'solar_energy',['solar_energy'],['load','offtake','injection'],lags,t, forecast_gap=forecast_gap, domain_min=[None, 0, 0, 0, None], domain_max=[None, 1, 1, 1, None])
    
    _, X_test_opt_lstm, _, _, scalers_opt = tensors_opt.create_tensor()
    _, X_test_opt_cvx, _, _, _ = tensors_opt.create_tensor()
    _, X_test_opt_naive, _, _, _ = tensors_opt.create_tensor()
    _, X_test_opt_perfect, _, _, _ = tensors_opt.create_tensor()
    
    initial_bat_tensor_test_lstm = torch.zeros([X_test_opt_lstm.shape[0],lags,1])
    initial_bat_tensor_test_cvx = torch.zeros([X_test_opt_cvx.shape[0],lags,1])
    initial_bat_tensor_test_naive = torch.zeros([X_test_opt_naive.shape[0],lags,1])
    initial_bat_tensor_test_perfect = torch.zeros([X_test_opt_perfect.shape[0],lags,1])
    
    if forecast_gap == 0:
        initial_bat_tensor_test_lstm[:,-1,:] = battery_capacity * 0.5
        initial_bat_tensor_test_cvx[:,-1,:] = battery_capacity * 0.5
        initial_bat_tensor_test_naive[:,-1,:] = battery_capacity * 0.5
        initial_bat_tensor_test_perfect[:,-1,:] = battery_capacity * 0.5

    else:
        initial_bat_tensor_test_lstm[:,-1,:] = torch.tensor(np.array(initial_battery_lstm[forecast_gap-1])[:,1]).unsqueeze(-1) 
        initial_bat_tensor_test_cvx[:,-1,:] = torch.tensor(np.array(initial_battery_cvx[forecast_gap-1])[:,1]).unsqueeze(-1) 
        initial_bat_tensor_test_naive[:,-1,:] = torch.tensor(np.array(initial_battery_naive[forecast_gap-1])[:,1]).unsqueeze(-1) 
        initial_bat_tensor_test_perfect[:,-1,:] = torch.tensor(np.array(initial_battery_perfect[forecast_gap-1])[:,1]).unsqueeze(-1) 

    X_test_opt_lstm = torch.concat([X_test_opt_lstm, initial_bat_tensor_test_lstm],dim=-1)
    X_test_opt_cvx = torch.concat([X_test_opt_cvx, initial_bat_tensor_test_cvx],dim=-1)
    X_test_opt_naive = torch.concat([X_test_opt_naive, initial_bat_tensor_test_naive],dim=-1)
    X_test_opt_perfect = torch.concat([X_test_opt_perfect, initial_bat_tensor_test_perfect],dim=-1)

    
    # Create the models for PV forecasts
    lstm = LSTM(1,neurons,layers,t,0.5).to(device)
    lstm_opt = LSTMOPT(1,neurons,layers,t,0.5,problem,parameters,variables,scalers_opt[0]).to(device)
    
    lstm.load_state_dict(torch.load('../models/LSTM/building_' + str(building) + '_' + str(t) + 'h.pth'))
    lstm_opt.load_state_dict(torch.load('../models/LSTM_CVX/building_' + str(building) + '_' + str(t) + 'h.pth'))
    
    print('Forecasting PV')
    
    # Forecast the PV
    pv_lstm_test = lstm(X_test.to(device))
    
    pv_sol_test, _ = lstm_opt(X_test_opt_cvx[:,:,0].unsqueeze(-1).to(device), 
                              X_test_opt_cvx[:,-t:,1].to(device), 
                              X_test_opt_cvx[:,-t:,2].to(device), 
                              X_test_opt_cvx[:,-t:,3].to(device), 
                              X_test_opt_cvx[:,-1,4].to(device))

    lstm_pv.append(torch_py(pv_lstm_test[:,0]))
    cvx_pv.append(torch_py(pv_sol_test[:,0]))


    # Obtain the initial energy for the next timeslot    
    initial_battery_lstm_t = []
    imported_energy_lstm_t = []
    exported_energy_lstm_t = []
    imported_energy_lstm_t_real = []
    exported_energy_lstm_t_real = []
        
    initial_battery_cvx_t = []
    imported_energy_cvx_t = []
    exported_energy_cvx_t = []
    imported_energy_cvx_t_real = []
    exported_energy_cvx_t_real = []
    
    initial_battery_naive_t = []
    imported_energy_naive_t = []
    exported_energy_naive_t = []
    imported_energy_naive_t_real = []
    exported_energy_naive_t_real = []
    
    initial_battery_perfect_t = []
    imported_energy_perfect_t = []
    exported_energy_perfect_t = []
    

    
    print('Obtaining optimization parameters for hour ' + str(forecast_gap+1))

    for j in range(len(X_test)):   
        # LSTM forecast
        parameters[0].value = torch_py(rescale(pv_lstm_test[j], scalers_opt[0]))
        parameters[1].value = torch_py(X_test_opt_lstm[j,-t:,1])
        parameters[2].value = torch_py(X_test_opt_lstm[j,-t:,2])
        parameters[3].value = torch_py(X_test_opt_lstm[j,-t:,3])
        parameters[4].value = torch_py(X_test_opt_lstm[j,-1:,4])[0]
        problem.solve()
        
        # LSTM real
        parameters_post[0].value = torch_py(rescale(y_test[j,:], scalers[0]))
        parameters_post[1].value = torch_py(X_test_opt_lstm[j,-t:,1])
        parameters_post[2].value = torch_py(X_test_opt_lstm[j,-t:,2])
        parameters_post[3].value = torch_py(X_test_opt_lstm[j,-t:,3])
        parameters_post[4].value = torch_py(X_test_opt_lstm[j,-1:,4])[0]
        parameters_post[5].value = variables[3].value
        parameters_post[6].value = variables[4].value    
        problem_post.solve()
        
        initial_battery_lstm_t.append(variables_post[2].value)
        imported_energy_lstm_t.append(variables[0].value)
        exported_energy_lstm_t.append(variables[1].value)
        imported_energy_lstm_t_real.append(variables_post[0].value)
        exported_energy_lstm_t_real.append(variables_post[1].value)
        
        
        # CVX forecast
        parameters[0].value = torch_py(rescale(pv_sol_test[j], scalers_opt[0]))
        parameters[1].value = torch_py(X_test_opt_cvx[j,-t:,1])
        parameters[2].value = torch_py(X_test_opt_cvx[j,-t:,2])
        parameters[3].value = torch_py(X_test_opt_cvx[j,-t:,3])
        parameters[4].value = torch_py(X_test_opt_cvx[j,-1:,4])[0]
        problem.solve()
    
        # CVX real
        parameters_post[0].value = torch_py(rescale(y_test[j,:], scalers[0]))
        parameters_post[1].value = torch_py(X_test_opt_cvx[j,-t:,1])
        parameters_post[2].value = torch_py(X_test_opt_cvx[j,-t:,2])
        parameters_post[3].value = torch_py(X_test_opt_cvx[j,-t:,3])
        parameters_post[4].value = torch_py(X_test_opt_cvx[j,-1:,4])[0]
        parameters_post[5].value = variables[3].value
        parameters_post[6].value = variables[4].value    
        problem_post.solve()
        
        initial_battery_cvx_t.append(variables_post[2].value)
        imported_energy_cvx_t.append(variables[0].value)
        exported_energy_cvx_t.append(variables[1].value)
        imported_energy_cvx_t_real.append(variables_post[0].value)
        exported_energy_cvx_t_real.append(variables_post[1].value)
        
        
        # Naive forecast 
        parameters[0].value = torch_py(rescale(X_test[j,forecast_gap:T,0], scalers_opt[0]))
        parameters[1].value = torch_py(X_test_opt_naive[j,-t:,1])
        parameters[2].value = torch_py(X_test_opt_naive[j,-t:,2])
        parameters[3].value = torch_py(X_test_opt_naive[j,-t:,3])
        parameters[4].value = torch_py(X_test_opt_naive[j,-1:,4])[0]
        problem.solve()
    
        # Naive real
        parameters_post[0].value = torch_py(rescale(y_test[j,:], scalers[0]))
        parameters_post[1].value = torch_py(X_test_opt_naive[j,-t:,1])
        parameters_post[2].value = torch_py(X_test_opt_naive[j,-t:,2])
        parameters_post[3].value = torch_py(X_test_opt_naive[j,-t:,3])
        parameters_post[4].value = torch_py(X_test_opt_naive[j,-1:,4])[0]
        parameters_post[5].value = variables[3].value
        parameters_post[6].value = variables[4].value    
        problem_post.solve()
        
        initial_battery_naive_t.append(variables_post[2].value)
        imported_energy_naive_t.append(variables[0].value)
        exported_energy_naive_t.append(variables[1].value)
        imported_energy_naive_t_real.append(variables_post[0].value)
        exported_energy_naive_t_real.append(variables_post[1].value)
        
        
        # Perfect forecast
        parameters[0].value = torch_py(rescale(y_test[j,:], scalers[0]))
        parameters[1].value = torch_py(X_test_opt_perfect[j,-t:,1])
        parameters[2].value = torch_py(X_test_opt_perfect[j,-t:,2])
        parameters[3].value = torch_py(X_test_opt_perfect[j,-t:,3])
        parameters[4].value = torch_py(X_test_opt_perfect[j,-1:,4])[0]
        problem.solve()
        
        initial_battery_perfect_t.append(variables_post[2].value)
        imported_energy_perfect_t.append(variables[0].value)
        exported_energy_perfect_t.append(variables[1].value)


    initial_battery_lstm.append(initial_battery_lstm_t)
    imported_energy_lstm.append(imported_energy_lstm_t)
    exported_energy_lstm.append(exported_energy_lstm_t)
    imported_energy_lstm_real.append(imported_energy_lstm_t_real)
    exported_energy_lstm_real.append(exported_energy_lstm_t_real)
    
    initial_battery_cvx.append(initial_battery_cvx_t)
    imported_energy_cvx.append(imported_energy_cvx_t)
    exported_energy_cvx.append(exported_energy_cvx_t)
    imported_energy_cvx_real.append(imported_energy_cvx_t_real)
    exported_energy_cvx_real.append(exported_energy_cvx_t_real)
    
    initial_battery_naive.append(initial_battery_naive_t)
    imported_energy_naive.append(imported_energy_naive_t)
    exported_energy_naive.append(exported_energy_naive_t)
    imported_energy_naive_real.append(imported_energy_naive_t_real)
    exported_energy_naive_real.append(exported_energy_naive_t_real)
    
    initial_battery_perfect.append(initial_battery_perfect_t)
    imported_energy_perfect.append(imported_energy_perfect_t)
    exported_energy_perfect.append(exported_energy_perfect_t)
    
    lags += 1
    forecast_gap += 1

Setting up optimization for hour 24
Forecasting PV
Obtaining optimization parameters for hour 1


C:\Users\jdepoort\Anaconda3\lib\site-packages\cvxpy\reductions\solvers\solving_chain.py:336: FutureWarning: 
    Your problem is being solved with the ECOS solver by default. Starting in 
    CVXPY 1.5.0, Clarabel will be used as the default solver instead. To continue 
    using ECOS, specify the ECOS solver explicitly using the ``solver=cp.ECOS`` 
    argument to the ``problem.solve`` method.
    
  warnings.warn(ECOS_DEPRECATION_MSG, FutureWarning)


In [8]:
imp_lstm = torch.tensor(imported_energy_lstm[0])
exp_lstm = torch.tensor(exported_energy_lstm[0])
imp_lstm_real = torch.tensor(imported_energy_lstm_real[0])
exp_lstm_real = torch.tensor(exported_energy_lstm_real[0])

imp_cvx = torch.tensor(imported_energy_cvx[0])
exp_cvx = torch.tensor(exported_energy_cvx[0])
imp_cvx_real = torch.tensor(imported_energy_cvx_real[0])
exp_cvx_real = torch.tensor(exported_energy_cvx_real[0])

imp_naive = torch.tensor(imported_energy_naive[0])
exp_naive = torch.tensor(exported_energy_naive[0])
imp_naive_real = torch.tensor(imported_energy_naive_real[0])
exp_naive_real = torch.tensor(exported_energy_naive_real[0])

imp_perfect = torch.tensor(imported_energy_perfect[0])
exp_perfect = torch.tensor(exported_energy_perfect[0])

C:\Users\jdepoort\AppData\Local\Temp\ipykernel_13748\886433427.py:1: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_new.cpp:264.)
  imp_lstm = torch.tensor(imported_energy_lstm[0])


RuntimeError: Could not infer dtype of NoneType

In [ ]:
tensors_opt = tensor.Tensors(pvb_system.house,'solar_energy',['solar_energy'],['load','offtake','injection'],24,T, domain_min=[None, 0, 0, 0, None], domain_max=[None, 1, 1, 1, None])
X_train_opt, X_test_opt, _, _, _ = tensors_opt.create_tensor()

In [ ]:
cost_lstm = imp_lstm * X_test_opt[:,:,2] - exp_lstm * X_test_opt[:,:,3]
cost_lstm_real = imp_lstm_real * X_test_opt[:,:,2] - exp_lstm_real * X_test_opt[:,:,3]

cost_cvx = imp_cvx * X_test_opt[:,:,2] - exp_cvx * X_test_opt[:,:,3]
cost_cvx_real = imp_cvx_real * X_test_opt[:,:,2] - exp_cvx_real * X_test_opt[:,:,3]

cost_naive = imp_naive * X_test_opt[:,:,2] - exp_naive * X_test_opt[:,:,3]
cost_naive_real = imp_naive_real * X_test_opt[:,:,2] - exp_naive_real * X_test_opt[:,:,3]

perfect = imp_perfect * X_test_opt[:,:,2] - exp_perfect * X_test_opt[:,:,3]

In [ ]:
print(f"The cost LSTM predicted: {torch.sum(cost_lstm):.2f}\nThe real cost with LSTM: {torch.sum(cost_lstm_real):.2f}\n\n"
      f"The cost CVX predicted: {torch.sum(cost_cvx):.2f}\nThe real cost with CVX: {torch.sum(cost_cvx_real):.2f}\n\n"
      f"The cost Naive predicted: {torch.sum(cost_naive):.2f}\nThe real cost with Naive: {torch.sum(cost_naive_real):.2f}\n\n"
      f"The cost for a perfect prediction: {torch.sum(perfect):.2f}\n")

# PLAYGROUND